In [ ]:
import pandas as pd
import sqlite3
import os
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
conn = sqlite3.connect('northwind.db')

caminho_dados = 'data/'

arquivos = [
    'categories', 'customer_customer_demo', 'customer_demographics', 
    'customers', 'employee_territories', 'employees', 'order_details', 
    'orders', 'products', 'region', 'shippers', 'suppliers', 
    'territories', 'us_states'
]

### O Fim das Planilhas Manuais

**O Problema:** A dependência de planilhas criava gargalos.

**A Solução:** Toda a manipulação, desde a extração até o cálculo do Ticket Médio, foi automatizada utilizando SQL e Python. Isso significa que este relatório não é estático; ele é um script reprodutível. À medida que novos dados entram no banco, basta reexecutar o processo e os KPIs e gráficos são atualizados instantaneamente, eliminando o trabalho manual e o risco de erros de digitação.

In [ ]:
for arquivo in arquivos:
    caminho_completo = f'{caminho_dados}{arquivo}.csv'
    df = pd.read_csv(caminho_completo, sep=';')
    df.to_sql(arquivo, conn, if_exists='replace', index=False)
print(f'{arquivos} importados com sucesso!')

['categories', 'customer_customer_demo', 'customer_demographics', 'customers', 'employee_territories', 'employees', 'order_details', 'orders', 'products', 'region', 'shippers', 'suppliers', 'territories', 'us_states'] importados com sucesso!


In [ ]:
print("--- Relatório de Auditoria de Dados (Raw Data) ---")

todas_tabelas = [
    'categories', 'customer_customer_demo', 'customer_demographics', 
    'customers', 'employee_territories', 'employees', 'order_details', 
    'orders', 'products', 'region', 'shippers', 'suppliers', 
    'territories', 'us_states'
]

for tabela in todas_tabelas:
    df_temp = pd.read_sql_query(f"SELECT * FROM {tabela}", conn)
    
    total_linhas = len(df_temp)
    total_nulos = df_temp.isnull().sum().sum()
    
    if total_linhas == 0:
        status = "❌ Tabela Vazia!"
    elif total_nulos == 0:
        status = "✅ Zero Nulos"
    else:
        status = f"Atenção: {total_nulos} valores nulos encontrados"
        
    nome_tabela = f"'{tabela.upper()}'".ljust(25)
    print(f"Tabela {nome_tabela} : {total_linhas} registros | Status: {status}")

--- Relatório de Auditoria de Dados (Raw Data) ---
Tabela 'CATEGORIES'              : 8 registros | Status: ✅ Zero Nulos
Tabela 'CUSTOMER_CUSTOMER_DEMO'  : 0 registros | Status: ❌ Tabela Vazia!
Tabela 'CUSTOMER_DEMOGRAPHICS'   : 0 registros | Status: ❌ Tabela Vazia!
Tabela 'CUSTOMERS'               : 91 registros | Status: Atenção: 83 valores nulos encontrados
Tabela 'EMPLOYEE_TERRITORIES'    : 49 registros | Status: ✅ Zero Nulos
Tabela 'EMPLOYEES'               : 9 registros | Status: Atenção: 5 valores nulos encontrados
Tabela 'ORDER_DETAILS'           : 2155 registros | Status: ✅ Zero Nulos
Tabela 'ORDERS'                  : 830 registros | Status: Atenção: 547 valores nulos encontrados
Tabela 'PRODUCTS'                : 77 registros | Status: ✅ Zero Nulos
Tabela 'REGION'                  : 4 registros | Status: ✅ Zero Nulos
Tabela 'SHIPPERS'                : 6 registros | Status: ✅ Zero Nulos
Tabela 'SUPPLIERS'               : 29 registros | Status: Atenção: 60 valores nulos encont

### *Single Source of Truth* (Fonte Única da Verdade)

**O Desafio:** Atualmente, a Northwind sofre com dados de diferentes áreas que não batem durante as reuniões. Isso ocorre pela falta de uma centralização das regras de negócio.

**A Solução Técnica:** Para resolver isso de forma escalável, o primeiro passo é criar uma **Tabela Fato de Vendas**. Essa tabela une os cabeçalhos dos pedidos (`orders`) com os itens comprados (`order_details`) e já calcula o **Valor Total Faturado** por item (descontando promoções). 

In [ ]:
query_fato_vendas = """
SELECT 
    o.order_id,
    o.customer_id,
    o.employee_id,
    DATE(o.order_date) AS data_pedido,
    od.product_id,
    od.quantity AS quantidade,
    od.unit_price AS preco_unitario,
    od.discount AS desconto,
    (od.quantity * od.unit_price * (1 - od.discount)) AS valor_total_item
FROM orders o
JOIN order_details od ON o.order_id = od.order_id
"""

In [ ]:
df_fato_vendas = pd.read_sql_query(query_fato_vendas, conn)
display(df_fato_vendas.head())

,order_id,customer_id,employee_id,data_pedido,product_id,quantidade,preco_unitario,desconto,valor_total_item
0,10248,VINET,5,1996-07-04,11,12,14.0,0.0,168.0
1,10248,VINET,5,1996-07-04,42,10,9.8,0.0,98.0
2,10248,VINET,5,1996-07-04,72,5,34.8,0.0,174.0
3,10249,TOMSP,6,1996-07-05,14,9,18.6,0.0,167.4
4,10249,TOMSP,6,1996-07-05,51,40,42.4,0.0,1696.0


### Objetivo Estratégico: Análise do Ticket Médio e Faturamento

**O Desafio:** A diretoria definiu como meta o aumento do ticket médio.

**A Solução Analítica:** Utilizando a nossa Fonte Única da Verdade, calculamos o Ticket Médio Global (Faturamento Total / Quantidade de Pedidos) e a sua evolução mensal. Isso nos permite identificar se o valor médio gasto pelos clientes está crescendo organicamente ou se está estagnado, além de dar ao CEO a visão financeira consolidada que faltava.

In [ ]:
query_ticket_medio = """
WITH metricas_pedidos AS (
    SELECT 
        o.order_id,
        STRFTIME('%Y-%m', o.order_date) AS mes_ano,
        SUM(od.quantity * od.unit_price * (1 - od.discount)) AS valor_total_pedido
    FROM orders o
    JOIN order_details od ON o.order_id = od.order_id
    GROUP BY o.order_id, mes_ano
)
SELECT 
    mes_ano,
    COUNT(DISTINCT order_id) AS total_pedidos,
    ROUND(SUM(valor_total_pedido), 2) AS faturamento_mensal,
    ROUND(SUM(valor_total_pedido) / COUNT(DISTINCT order_id), 2) AS ticket_medio
FROM metricas_pedidos
GROUP BY mes_ano
ORDER BY mes_ano;
"""

In [ ]:
df_ticket_medio = pd.read_sql_query(query_ticket_medio, conn)
display(df_ticket_medio)

,mes_ano,total_pedidos,faturamento_mensal,ticket_medio
0,1996-07,22,27861.90,1266.45
1,1996-08,25,25485.28,1019.41
2,1996-09,23,26381.40,1147.02
3,1996-10,26,37515.72,1442.91
4,1996-11,25,45600.04,1824.00
5,1996-12,31,45239.63,1459.34
6,1997-01,33,61258.07,1856.31
7,1997-02,29,38483.64,1327.02
8,1997-03,30,38547.22,1284.91
9,1997-04,31,53032.95,1710.74


In [ ]:
faturamento_total = df_ticket_medio['faturamento_mensal'].sum()
total_pedidos_global = df_ticket_medio['total_pedidos'].sum()
ticket_medio_global = faturamento_total / total_pedidos_global

print(f"💰 Faturamento Total Histórico: ${faturamento_total:,.2f} USD")
print(f"📦 Total de Pedidos Realizados: {total_pedidos_global}")
print(f"🛒 Ticket Médio Global: ${ticket_medio_global:,.2f} USD")

💰 Faturamento Total Histórico: $1,265,793.04 USD
📦 Total de Pedidos Realizados: 830
🛒 Ticket Médio Global: $1,525.05 USD


### Visualização de Desempenho: Evolução do Ticket Médio

**O Foco no Comercial:** A diretoria comercial precisa acompanhar a saúde das vendas de forma clara. O gráfico abaixo ilustra a variação do Ticket Médio ao longo dos meses. 

**Insight Preliminar:** Ao visualizar essa linha do tempo, podemos identificar sazonalidades e avaliar se o valor médio dos pedidos está estagnado, o que justificaria uma nova campanha de *cross-sell* para incentivar compras casadas.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_ticket_medio['mes_ano'],
    y=df_ticket_medio['ticket_medio'],
    mode='lines+markers',
    name='Ticket Médio',
    line=dict(color='#00D9FF', width=3),
    marker=dict(size=10, color='#00D9FF', symbol='circle',
                line=dict(color='#0099CC', width=2)),
    hovertemplate='<b>%{x}</b><br>Ticket Médio: $%{y:.2f} USD<extra></extra>'
))

fig.add_trace(go.Scatter(
    x=df_ticket_medio['mes_ano'],
    y=df_ticket_medio['ticket_medio'],
    fill='tozeroy',
    fillcolor='rgba(0, 217, 255, 0.1)',
    line=dict(width=0),
    showlegend=False,
    hoverinfo='skip'
))

fig.update_layout(
    title={
        'text': '📈 Evolução Mensal do Ticket Médio',
        'font': {'size': 24, 'color': '#1f1f1f', 'family': 'Arial Black'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Período',
    yaxis_title='Ticket Médio (USD)',
    font=dict(family='Arial', size=12, color='#333'),
    hovermode='x unified',
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=500,
    margin=dict(l=60, r=40, t=80, b=80),
    xaxis=dict(
        showgrid=True,
        gridcolor='#E5E5E5',
        tickangle=-45,
        showline=True,
        linecolor='#CCCCCC'
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='#E5E5E5',
        showline=True,
        linecolor='#CCCCCC',
        rangemode='tozero'
    ),
    hoverlabel=dict(
        bgcolor="white",
        font_size=13,
        font_family="Arial"
    )
)

max_idx = df_ticket_medio['ticket_medio'].idxmax()

annotations = [
    # Primeiro ponto
    dict(
        x=df_ticket_medio['mes_ano'].iloc[0],
        y=df_ticket_medio['ticket_medio'].iloc[0],
        text=f"${df_ticket_medio['ticket_medio'].iloc[0]:.0f}",
        showarrow=True,
        arrowhead=2,
        arrowcolor='#00D9FF',
        ax=0,
        ay=-40,
        font=dict(size=11, color='#00D9FF', family='Arial Black')
    ),
    # Ponto máximo
    dict(
        x=df_ticket_medio['mes_ano'].iloc[max_idx],
        y=df_ticket_medio['ticket_medio'].iloc[max_idx],
        text=f"Pico: ${df_ticket_medio['ticket_medio'].iloc[max_idx]:.0f}",
        showarrow=True,
        arrowhead=2,
        arrowcolor='#FF6B35',
        ax=0,
        ay=-50,
        font=dict(size=12, color='#FF6B35', family='Arial Black')
    ),
    # Último ponto
    dict(
        x=df_ticket_medio['mes_ano'].iloc[-1],
        y=df_ticket_medio['ticket_medio'].iloc[-1],
        text=f"${df_ticket_medio['ticket_medio'].iloc[-1]:.0f}",
        showarrow=True,
        arrowhead=2,
        arrowcolor='#00D9FF',
        ax=0,
        ay=-40,
        font=dict(size=11, color='#00D9FF', family='Arial Black')
    )
]

fig.update_layout(annotations=annotations)

fig.show()

### 💡 Insights do Dashboard de Performance

**Categorias:** Identificar quais linhas de produtos geram mais receita permite otimizar estoque e direcionar campanhas de marketing para as categorias de maior valor.

**Geografia:** Conhecer os mercados mais rentáveis ajuda a definir prioridades de expansão, logística e estratégias de precificação regional.

**Vendedores:** Reconhecer os top performers possibilita replicar suas melhores práticas através de treinamentos e programas de mentoria, além de fundamentar políticas de bonificação baseadas em dados.

**Ação Recomendada:** Cruzar essas três dimensões (ex: quais vendedores vendem mais em quais países e categorias) pode revelar oportunidades ocultas de otimização comercial.

In [ ]:
#Top 5 Categorias por Faturamento
query_categorias = """
SELECT 
    c.category_name as categoria,
    ROUND(SUM(od.quantity * od.unit_price * (1 - od.discount)), 2) as faturamento
FROM order_details od
JOIN products p ON od.product_id = p.product_id
JOIN categories c ON p.category_id = c.category_id
JOIN orders o ON od.order_id = o.order_id
GROUP BY c.category_name
ORDER BY faturamento DESC
LIMIT 5
"""

#Top 5 Países por Faturamento
query_paises = """
SELECT 
    o.ship_country as pais,
    ROUND(SUM(od.quantity * od.unit_price * (1 - od.discount)), 2) as faturamento
FROM orders o
JOIN order_details od ON o.order_id = od.order_id
GROUP BY o.ship_country
ORDER BY faturamento DESC
LIMIT 5
"""

#Top 5 Vendedores por Faturamento
query_vendedores = """
SELECT 
    e.first_name || ' ' || e.last_name as vendedor,
    ROUND(SUM(od.quantity * od.unit_price * (1 - od.discount)), 2) as faturamento
FROM orders o
JOIN order_details od ON o.order_id = od.order_id
JOIN employees e ON o.employee_id = e.employee_id
GROUP BY vendedor
ORDER BY faturamento DESC
LIMIT 5
"""

df_categorias = pd.read_sql_query(query_categorias, conn)
df_paises = pd.read_sql_query(query_paises, conn)
df_vendedores = pd.read_sql_query(query_vendedores, conn)


In [ ]:
from plotly.subplots import make_subplots

fig_dashboard = make_subplots(
    rows=1, cols=3,
    subplot_titles=('💰 Top 5 Categorias', '🌍 Top 5 Países', '👥 Top 5 Vendedores'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}, {'type': 'bar'}]],
    horizontal_spacing=0.12
)

# Gráfico 1: Categorias 
fig_dashboard.add_trace(
    go.Bar(
        y=df_categorias['categoria'][::-1],  # Inverter para maior no topo
        x=df_categorias['faturamento'][::-1],
        orientation='h',
        marker=dict(color='#00D9FF', line=dict(color='#0099CC', width=1)),
        text=df_categorias['faturamento'][::-1].apply(lambda x: f'${x/1000:.0f}K'),
        textposition='inside',
        showlegend=False,
        hovertemplate='<b>%{y}</b><br>Faturamento: $%{x:,.0f}<extra></extra>'
    ),
    row=1, col=1
)

# Gráfico 2: Países
fig_dashboard.add_trace(
    go.Bar(
        x=df_paises['pais'],
        y=df_paises['faturamento'],
        marker=dict(
            color=df_paises['faturamento'],
            colorscale='Blues',
            showscale=False,
            line=dict(color='#0099CC', width=1)
        ),
        text=df_paises['faturamento'].apply(lambda x: f'${x/1000:.0f}K'),
        textposition='outside',
        showlegend=False,
        hovertemplate='<b>%{x}</b><br>Faturamento: $%{y:,.0f}<extra></extra>'
    ),
    row=1, col=2
)

# Gráfico 3: Vendedores
fig_dashboard.add_trace(
    go.Bar(
        x=df_vendedores['vendedor'],
        y=df_vendedores['faturamento'],
        marker=dict(
            color=df_vendedores['faturamento'],
            colorscale='Viridis',
            showscale=False,
            line=dict(color='#444', width=1)
        ),
        text=df_vendedores['faturamento'].apply(lambda x: f'${x/1000:.0f}K'),
        textposition='outside',
        showlegend=False,
        hovertemplate='<b>%{x}</b><br>Faturamento: $%{y:,.0f}<extra></extra>'
    ),
    row=1, col=3
)

# Atualizar layout geral
fig_dashboard.update_layout(
    title={
        'text': '📊 Dashboard de Performance: Principais Drivers de Receita',
        'font': {'size': 24, 'color': '#1f1f1f', 'family': 'Arial Black'},
        'x': 0.5,
        'xanchor': 'center'
    },
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=500,
    margin=dict(l=20, r=80, t=120, b=80),  # Aumentado: r=80 (direita), t=120 (topo)
    font=dict(family='Arial', size=11, color='#333')
)

fig_dashboard.update_xaxes(showgrid=True, gridcolor='#E5E5E5', showline=True, linecolor='#CCC')
fig_dashboard.update_yaxes(showgrid=True, gridcolor='#E5E5E5', showline=True, linecolor='#CCC')

fig_dashboard.update_xaxes(tickangle=-45, row=1, col=2)

fig_dashboard.update_xaxes(tickangle=-45, row=1, col=3)

# Título dos eixos
fig_dashboard.update_xaxes(title_text="Faturamento (USD)", row=1, col=1)
fig_dashboard.update_xaxes(title_text="País", row=1, col=2)
fig_dashboard.update_xaxes(title_text="Vendedor", row=1, col=3)

fig_dashboard.update_yaxes(title_text="Categoria", row=1, col=1)
fig_dashboard.update_yaxes(title_text="Faturamento (USD)", row=1, col=2)
fig_dashboard.update_yaxes(title_text="Faturamento (USD)", row=1, col=3)

fig_dashboard.show()

### Análise e Redução de Churn

**O Desafio:** A diretoria definiu a redução do *churn* (evasão de clientes) como prioridade. Como a Northwind Traders não atua com um modelo de assinatura, precisamos definir matematicamente o que significa "perder um cliente".

**A Regra de Negócio (Visão Analítica):** Levando em consideração que o core business da empresa é a venda de bens de consumo rápido (alimentos, bebidas e utilidades domésticas), o ciclo de recompra é curto. Portanto, foi definido que **um cliente é considerado em "Churn" se a sua última compra ocorreu há mais de 3 meses (90 dias)** em relação à data atual do sistema.

**A Solução Técnica:** Utilizando SQL, calculamos a data da última compra de cada cliente e a comparamos com a data da última transação registrada na empresa, segmentando a base entre "Ativos" e "Churn".

In [ ]:
query_churn = """
WITH data_sistema AS (
    SELECT MAX(DATE(order_date)) as data_atual FROM orders
),
ultima_compra_cliente AS (
    SELECT 
        customer_id, 
        MAX(DATE(order_date)) as ultima_compra
    FROM orders
    GROUP BY customer_id
),
classificacao_clientes AS (
    SELECT 
        u.customer_id,
        u.ultima_compra,
        d.data_atual,
        CAST(JULIANDAY(d.data_atual) - JULIANDAY(u.ultima_compra) AS INTEGER) AS dias_inativo,
        CASE 
            WHEN CAST(JULIANDAY(d.data_atual) - JULIANDAY(u.ultima_compra) AS INTEGER) > 90 THEN 'Churn'
            ELSE 'Ativo'
        END AS status
    FROM ultima_compra_cliente u
    CROSS JOIN data_sistema d
)
SELECT 
    status,
    COUNT(customer_id) AS total_clientes,
    ROUND(COUNT(customer_id) * 100.0 / (SELECT COUNT(*) FROM classificacao_clientes), 2) AS percentual
FROM classificacao_clientes
GROUP BY status;
"""

In [ ]:
df_churn = pd.read_sql_query(query_churn, conn)

In [ ]:
fig_churn = go.Figure()

colors = ['#00D9FF', '#FF6B35'] 

fig_churn.add_trace(go.Bar(
    x=df_churn['status'],
    y=df_churn['total_clientes'],
    text=df_churn.apply(lambda row: f"{row['total_clientes']}<br>({row['percentual']}%)", axis=1),
    textposition='outside',
    marker=dict(
        color=colors,
        line=dict(color='#1f1f1f', width=2)
    ),
    hovertemplate='<b>%{x}</b><br>Clientes: %{y}<br><extra></extra>'
))

fig_churn.update_layout(
    title={
        'text': '🚨 Análise de Churn: Base de Clientes Ativos vs Inativos',
        'font': {'size': 22, 'color': '#1f1f1f', 'family': 'Arial Black'},
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Status do Cliente',
    yaxis_title='Quantidade de Clientes',
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=450,
    showlegend=False,
    yaxis=dict(rangemode='tozero')
)

fig_churn.show()

### Plano de Ação (Para Inovação & Comercial)

Com uma taxa de *Churn* de aproximadamente **18%** (16 clientes inativos na janela de 90 dias) e a necessidade de aumentar o Ticket Médio, sugiro as seguintes frentes de inovação *data-driven*:

1. **Campanha de Reativação (Win-back):** O time comercial agora possui a lista exata dos 16 clientes inativos. Sugerimos uma abordagem focada e personalizada, oferecendo condições especiais (como frete grátis ou descontos nas categorias que eles mais compravam) para recuperar essa receita adormecida.
2. **Combos Inteligentes (Cross-sell):** Para maximizar o valor dos **73 clientes ativos**, podemos utilizar o banco de dados para estruturar uma *Market Basket Analysis* (Análise de Cesta de Compras), descobrindo quais produtos saem juntos com frequência. Com isso, criamos "Combos" estratégicos, forçando o aumento orgânico do Ticket Médio.

---

### 🏛️ Recomendação de Arquitetura (Para a TI)

Entendemos os receios históricos da TI com projetos de BI caros e demorados. Para garantir que os dados "sempre batam" sem estourar o orçamento, recomendamos a adoção do **Modern Data Stack (MDS)**:

* **Ingestão e Armazenamento:** Manter o banco PostgreSQL em nuvem como nosso *Data Warehouse* central. Utilizar ferramentas open-source ou de baixo custo para centralizar os dados do ERP, Salesforce e ContaAzul.
* **Transformação (O Segredo):** Utilizar o **dbt (data build tool)**. Em vez de criar views complexas ou planilhas pesadas, o dbt transforma os dados brutos na nossa `Fato_Vendas` de forma versionada (via Git), documentada e com testes de qualidade automáticos. Se um dado vier nulo, o dbt avisa antes de quebrar o painel do CEO.
* **Visualização:** Conectar uma ferramenta como PowerBI, Metabase ou Looker Studio diretamente nas tabelas limpas pelo dbt, garantindo velocidade e uma única versão da verdade.

## 📄 Exportação do Relatório em HTML

A célula abaixo gera um arquivo HTML completo e autocontido com todos os gráficos Plotly embutidos, pronto para apresentação ao CEO.

In [ ]:
# Gerar HTML completo com todos os gráficos embutidos
import plotly.io as pio

html_content = f"""
<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Relatório de Análise - Northwind Traders</title>
    <style>
        body {{
            font-family: 'Segoe UI', Arial, sans-serif;
            max-width: 1200px;
            margin: 0 auto;
            padding: 40px 20px;
            background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
            color: #333;
        }}
        .container {{
            background: white;
            padding: 40px;
            border-radius: 10px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.1);
        }}
        h1 {{
            color: #1f1f1f;
            border-bottom: 4px solid #00D9FF;
            padding-bottom: 15px;
            margin-bottom: 30px;
            font-size: 2.5em;
        }}
        h2 {{
            color: #0099CC;
            margin-top: 40px;
            font-size: 1.8em;
        }}
        h3 {{
            color: #333;
            margin-top: 30px;
            font-size: 1.3em;
        }}
        .kpi-box {{
            background: linear-gradient(135deg, #00D9FF 0%, #0099CC 100%);
            color: white;
            padding: 25px;
            border-radius: 8px;
            margin: 20px 0;
            box-shadow: 0 4px 15px rgba(0,217,255,0.3);
        }}
        .kpi-box h3 {{
            color: white;
            margin-top: 0;
        }}
        .kpi-value {{
            font-size: 2em;
            font-weight: bold;
            margin: 10px 0;
        }}
        .insight {{
            background: #f8f9fa;
            border-left: 4px solid #00D9FF;
            padding: 20px;
            margin: 20px 0;
            border-radius: 4px;
        }}
        .action-item {{
            background: #fff3cd;
            border-left: 4px solid #ffc107;
            padding: 15px;
            margin: 15px 0;
            border-radius: 4px;
        }}
        .chart-container {{
            margin: 30px 0;
        }}
        ul {{
            line-height: 1.8;
        }}
        .footer {{
            margin-top: 50px;
            padding-top: 20px;
            border-top: 2px solid #e0e0e0;
            text-align: center;
            color: #666;
        }}
    </style>
</head>
<body>
    <div class="container">
        <h1>📊 Relatório de Análise de Indicadores - Northwind Traders</h1>
        
        <div class="insight">
            <strong>Resumo Executivo:</strong> Este relatório apresenta uma análise completa dos principais indicadores de performance da Northwind Traders, incluindo Ticket Médio, Faturamento, Análise de Churn e identificação dos principais drivers de receita por categoria, geografia e vendedor.
        </div>

        <h2>💰 KPIs Principais</h2>
        <div class="kpi-box">
            <h3>Faturamento Total Histórico</h3>
            <div class="kpi-value">${faturamento_total:,.2f} USD</div>
        </div>
        <div class="kpi-box">
            <h3>Total de Pedidos Realizados</h3>
            <div class="kpi-value">{total_pedidos_global}</div>
        </div>
        <div class="kpi-box">
            <h3>Ticket Médio Global</h3>
            <div class="kpi-value">${ticket_medio_global:,.2f} USD</div>
        </div>

        <h2>📈 Evolução do Ticket Médio</h2>
        <div class="chart-container">
            {pio.to_html(fig, include_plotlyjs='cdn', full_html=False)}
        </div>

        <div class="insight">
            <strong>Insight:</strong> A análise temporal do ticket médio revela padrões sazonais importantes. O pico de <strong>${df_ticket_medio['ticket_medio'].max():.2f}</strong> indica oportunidades para replicar estratégias bem-sucedidas.
        </div>

        <h2>📊 Dashboard de Performance: Principais Drivers de Receita</h2>
        <div class="chart-container">
            {pio.to_html(fig_dashboard, include_plotlyjs='cdn', full_html=False)}
        </div>

        <div class="insight">
            <strong>Principais Descobertas:</strong>
            <ul>
                <li><strong>Top Categoria:</strong> {df_categorias.iloc[0]['categoria']} - ${df_categorias.iloc[0]['faturamento']:,.2f}</li>
                <li><strong>Top País:</strong> {df_paises.iloc[0]['pais']} - ${df_paises.iloc[0]['faturamento']:,.2f}</li>
                <li><strong>Top Vendedor:</strong> {df_vendedores.iloc[0]['vendedor']} - ${df_vendedores.iloc[0]['faturamento']:,.2f}</li>
            </ul>
        </div>

        <h2>🚨 Análise de Churn</h2>
        <div class="chart-container">
            {pio.to_html(fig_churn, include_plotlyjs='cdn', full_html=False)}
        </div>

        <div class="action-item">
            <strong>⚠️ Taxa de Churn:</strong> {df_churn[df_churn['status']=='Churn']['percentual'].values[0] if 'Churn' in df_churn['status'].values else 0}% 
            - Representa {df_churn[df_churn['status']=='Churn']['total_clientes'].values[0] if 'Churn' in df_churn['status'].values else 0} clientes inativos que precisam de atenção imediata.
        </div>

        <h2>🎯 Plano de Ação Recomendado</h2>
        <div class="action-item">
            <h3>1. Campanha de Reativação (Win-back)</h3>
            <p>Implementar campanha personalizada para os clientes inativos, oferecendo condições especiais nas categorias de maior interesse histórico.</p>
        </div>

        <div class="action-item">
            <h3>2. Otimização do Ticket Médio</h3>
            <p>Desenvolver combos inteligentes baseados em análise de cesta de compras para aumentar o valor médio dos pedidos.</p>
        </div>

        <div class="action-item">
            <h3>3. Foco Estratégico em Top Performers</h3>
            <p>Investir recursos em:</p>
            <ul>
                <li>Categorias de maior faturamento para otimização de estoque</li>
                <li>Mercados geográficos mais rentáveis para expansão</li>
                <li>Replicação de práticas dos vendedores de melhor desempenho</li>
            </ul>
        </div>

        <h2>🏛️ Próximos Passos - Arquitetura de Dados</h2>
        <div class="insight">
            <strong>Recomendação Técnica:</strong> Implementar Modern Data Stack (MDS) com dbt para transformação de dados, garantindo uma única fonte de verdade e reduzindo conflitos entre áreas.
        </div>

        <div class="footer">
            <p><strong>Northwind Traders - Relatório Confidencial</strong></p>
            <p>Gerado automaticamente em {pd.Timestamp.now().strftime('%d/%m/%Y às %H:%M')}</p>
        </div>
    </div>
</body>
</html>
"""

# Salvar o arquivo HTML
with open('relatorio_northwind_completo.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print("✅ Relatório HTML gerado com sucesso: relatorio_northwind_completo.html")
print("📄 Abra o arquivo no navegador para visualizar o relatório completo com todos os gráficos.")